# Runner Walkthrough Notebook

This notebook illustrates the training pipeline in `backend/src/runner/runner.py` step by step.

## Models used
- Logistic Regression
- Neural Network (MLPClassifier)
- Gradient Boosting (GradientBoostingClassifier)

## 1) Setup Imports and Paths
The notebook lives under `notebooks/`, so we add `../backend` to `sys.path`.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
BACKEND_DIR = (ROOT / '..' / 'backend').resolve() if ROOT.name == 'notebooks' else (ROOT / 'backend').resolve()
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print('Backend path:', BACKEND_DIR)

Backend path: /Users/shawnemersonmac/Desktop/untitled folder/backend


In [2]:
import config
import pandas as pd
from sklearn.model_selection import train_test_split

from src.architecture.data_pipeline import DataLoader, DataCleaner, FeatureEngineer
from src.architecture.ml_tasks import Evaluator, Predictor
from src.architecture.ml_utils import Converters, Pipeliner, ProfileGenerator
from src.models import LogisticRegressionModel, NeuralNetworkModel, GradientBoostingModel

## 2) Inspect the model objects and key hyperparameters

In [3]:
models = [LogisticRegressionModel(), NeuralNetworkModel(), GradientBoostingModel()]
for m in models:
    print(f"\n{m.get_name()}")
    print(m.model)


Logistic Regression
LogisticRegression(C=0.5, max_iter=1000, random_state=42)

Neural Network
MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=500, random_state=42)

Gradient Boosting
GradientBoostingClassifier(learning_rate=0.05, max_depth=4, min_samples_leaf=10,
                           n_estimators=300, random_state=42, subsample=0.8)


## 3) Load and clean data (same as Runner)

In [4]:
loader = DataLoader()
cleaner = DataCleaner()
engineer = FeatureEngineer()
converters = Converters()

raw = loader.load()
raw = loader.filter_consent(raw)

cols_needed = config.NUMERIC_COLS + config.CATEGORICAL_COLS + [config.TARGET]
df = raw[cols_needed].copy()
df = cleaner.clean(df, config.NUMERIC_COLS, config.CATEGORICAL_COLS, config.TARGET)
df = engineer.engineer(df, config.DERIVED_COLS, config.TARGET, converters, config.TARGET_CATEGORY)

print('Raw shape:', raw.shape)
print('Modeling dataframe shape:', df.shape)
df.head()

Raw shape: (2955, 28)
Modeling dataframe shape: (2842, 27)


,age,hours_work,social_media_use,rent,friends_count,highest_speed,dates,standard_drinks,countries,semesters,...,student_type,mainstream_advanced,lecture_mode,study_type,learner_style,stress,financial_pressure,work_study_ratio,social_engagement,stress_category
1,18.00,40.0,4.0,400.0,2.0,150.0,0.0,6.0,6.0,4.0,...,International,DATA1001,Live in the Lecture Theatre,I work steadily all semester,Style 1,10.0,10.000000,8.000000,0.500000,High (7-10)
2,19.00,40.0,22.5,200.8,0.0,0.0,0.0,4.3,6.0,8.3,...,International,DATA1901,Other,It changes depending on the subject,Style 3,1.0,5.020000,1.600000,0.000000,Low (1-3)
3,33.59,38.5,2.0,1000.0,100.0,140.0,0.0,0.0,6.0,0.0,...,Domestic,DATA1001,Other,I work steadily all semester,Style 1,5.0,25.974026,3.850000,33.333333,Average (4-6)
4,17.00,10.0,10.0,500.0,1.0,200.0,0.0,0.0,6.0,5.0,...,Domestic,DATA1001,Live in the Lecture Theatre,I work steadily all semester,Style 2,7.0,50.000000,3.333333,0.100000,High (7-10)
6,21.00,0.0,2.0,378.0,10.0,80.0,0.0,0.0,6.0,3.0,...,International,DATA1001,Live in the Lecture Theatre,I leave things to the last minute,Style 2,3.0,378.000000,0.000000,5.000000,Low (1-3)


## 4) Build features, split train/test, and preprocess

In [5]:
X_num = df[config.ALL_NUMERIC].copy()
X_cat = raw.loc[df.index, config.ALL_CATS].fillna('Unknown')
X = pd.concat([X_num, X_cat], axis=1)
y = df[config.TARGET_CATEGORY]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, random_state=config.SEED, stratify=y
)

pipeliner = Pipeliner(config.ALL_NUMERIC, config.ALL_CATS)
X_train = pipeliner.fit_transform(X_train_raw)
X_test = pipeliner.transform(X_test_raw)

print('Train size:', len(X_train))
print('Test size:', len(X_test))
print('Encoded feature count:', X_train.shape[1])

Train size: 2273
Test size: 569
Encoded feature count: 44


## 5) Train all models and evaluate

In [6]:
for model in models:
    model.train(X_train, y_train.values)

pred_cats = {m.get_name(): m.predict(X_test) for m in models}
evaluator = Evaluator()
reports = evaluator.classification_report_all(y_test.values, pred_cats, config.CATEGORY_ORDER)

for name, data in reports.items():
    print(f"{name:<22} accuracy: {data['accuracy']:.3f}")

Logistic Regression    accuracy: 0.429
Neural Network         accuracy: 0.364
Gradient Boosting      accuracy: 0.413


In [7]:
# Optional: full text reports
evaluator.print_classification_reports(reports)


  Logistic Regression  |  Accuracy: 42.9%
               precision    recall  f1-score   support

    Low (1-3)       0.44      0.26      0.33       155
Average (4-6)       0.44      0.67      0.53       239
  High (7-10)       0.40      0.25      0.30       175

     accuracy                           0.43       569
    macro avg       0.42      0.39      0.39       569
 weighted avg       0.42      0.43      0.40       569


  Neural Network  |  Accuracy: 36.4%
               precision    recall  f1-score   support

    Low (1-3)       0.31      0.30      0.30       155
Average (4-6)       0.42      0.43      0.42       239
  High (7-10)       0.34      0.34      0.34       175

     accuracy                           0.36       569
    macro avg       0.35      0.35      0.35       569
 weighted avg       0.36      0.36      0.36       569


  Gradient Boosting  |  Accuracy: 41.3%
               precision    recall  f1-score   support

    Low (1-3)       0.39      0.30      0.34  

## 6) Generate one sample profile and predict (same style as Runner)

In [8]:
profile_generator = ProfileGenerator(
    config.NUMERIC_COLS, config.CATEGORICAL_COLS, config.ALL_NUMERIC, config.ALL_CATS
)
predictor = Predictor()

sample = profile_generator.generate_profile(df, seed=config.SEED, mode='random')
sample_df = profile_generator.build(sample)
results = predictor.predict(sample_df, models, pipeliner)

sample_df

,age,hours_work,social_media_use,rent,friends_count,highest_speed,dates,standard_drinks,countries,semesters,...,work_study_ratio,social_engagement,gender,relationship_status,drug_use_ans,student_type,mainstream_advanced,lecture_mode,study_type,learner_style
0,29.839931,31.239367,19.318453,791.79166,20.0,261.818014,76.11397,38.194865,5.227036,4.503859,...,1.561968,1.03528,Non-binary / third gender,Its complicated,No,Domestic,DATA1001,Recording,I work steadily all semester,Style 1


In [9]:
predictor.print_results(results)
results

  STUDENT STRESS PREDICTION
  Logistic Regression   : High (7-10)
  Neural Network        : High (7-10)
  Gradient Boosting     : Average (4-6)


[{'model_name': 'Logistic Regression', 'category': 'High (7-10)'},
 {'model_name': 'Neural Network', 'category': 'High (7-10)'},
 {'model_name': 'Gradient Boosting', 'category': 'Average (4-6)'}]

## Notes
- This notebook mirrors `Runner.run()` without plotting-heavy EDA cells.
- To exactly match training artifacts, keep `config.SEED` and feature lists unchanged.
- If needed, add model saving cells (`model.save(...)`, `pipeliner.save(...)`) at the end.